## Agent sai ở đâu?

Agent: "Bitcoin khoảng 80000" -> không có tool call nào trước đây
Calculator: 0.5 * 80000 = 40000 -> tính đúng nhưng input là số bịa
Agent: 0.5 BTC trị giá 40000 -> sai vì input sai, dù logic cộng trừ đúng

Lỗi nằm ở bước đầu tiên: Agent tự phát biểu giá Bitcoin từ kiến thức nội tại thay vì gọi một tool tra giá thực tế. Mọi bước sau đều đúng về mặt tính toán nhưng vô nghĩa vì được xây dựng trên một con số không có căn cứ,

## Phân loại theo taxonomy
            
Planning failure        Có         Task tìm giá hiện tại và tính toán đòi hỏi 2 bước: (1) lấy giá real-time qua tool (2) tính toán. Agent lập kết hoạch thiếu b1
Tool selectiion failure Một phần   Không hẳn chọn sai, không chọn tool nào trong phần tra giá
Tool execution failure  Không      Calculator chạy đúng
Observation failure      Không     Agent đọc đúng output của calculator, không diễn giải sai kết quả tool trả về
Reasoning failure       Không      Phép nhân và câu kết luận về mặt logic là đùng - garbage in,garbage out, lỗi ko nằm ở suy luận
Hallucination           CÓ         80000 là con số bịa ra

Chuỗi nhân quả: Planning failure (biếu bước bắt buộc gọi tool cho dữ liệu real-time) -> tạo khoảng trống -> agent lấp khoảng trống đó bằng hallucination -> 2 bước sau (tool execution, reasoning) đều sạch về kĩ thuật nhưng kết thừa lỗi từ đầu vào sai. 

## Thiết kế lại dữ realtime bắt buôc qua tool

Nguyên tắc cốt lõi: agent không bao giờ đươc phép tự phát ra một con số thuộc nhóm volatile face trừ khi con số đó truy ngược được về một tool_result cụ thể trong trajectory.

1. Query Classifer (trước Planner)
- Phát hiện query có chứa "volatile fact" domain không:
    price / crypto / stock/ ...
- Nếu có -> gắn cờ requires_tool_ground = True cho domain đó

2. Planner (bị rằng buộc bởi cờ trên)
- Nếu requires_tool_grounding == True cho 1 domain nào đó trong task: plan bắt buộc phải chứa 1 bước gọi tool tương ứng (vd get_crypto_price) trước bất kì bước nào dùng con số đó.
- Planner không được phép tạo bước answer directly cho phần thuộc domain này

3. Execution + Grounding Validator (chạy sau mỗi lần Agent định sinh output)
- Trích các con số/claim thuộc domain volatitle trong câu trả lời dự kiến
- Đối chiếu: con số đó có xuất hiện trong bất kì tool_result nào của trajectory (có timestamp, có tool_call_id) không?
- Nếu không tìm thấy nguồn -> chặn, không cho publish, trả agent quay lại bước Planning với lỗi rõ ràng: "Cần gọi get_crypto_price trước khi trả lời"
- Nếu có nguồn -> cho phép, đồng thời gắn citation (tool_call_id) vào output để phục vụ observability/audit sau này

4. Calculator chỉ được nhận input đã có nguồn gốc từ bước 3, không được nhận giá trị agent tự đề xuất mà chưa qua validator.

Trajectory đúng phải trông như này:
Agent (plan): cần get_crypto_price("BTC") rồi Calculator
Tool[get_crypto_price]: BTC = $a (timestamp: ...)
Agent: Dùng $a từ tool_result
Calculator: 0.5 * a = ...
Agent: 0.5 BTC hiện trị giá... (theo giá trị tại {timestamp})

Điểm quan trọng khác:
- GHi rõ timestamp của giá trong câu trả lời cuối, vì giá crypto đổi liên tục - tránh ảo giác về độ chính xác nếu tool trả về giá đã cache lâu
- Grounding Validator nên là một bước tách khỏi chính LLM sinh câu trả lời (rule-based regex tìm pattern tiền tệ/số + domain keyword), không nên tự LLM  tự kiểm tra chính mình vì cùng một lỗi hallucination có thể lặp lại ở bước tự kiếm

## Trả lời 5 câu hỏi bắt buộc

1. Failure mode lớn nhât: Hallucination dữ liệu volatile (giá/thời gian) được trình bày với độ tự tin như fact đã kiểm chứng - nguy hiểm vì downstream tool (Calculator) làm nó trông có vẻ đáng tin cho phép tính đúng

2. Biết hệ thống tốt lên khi nào? 
Tỷ lệ trajectory có claim volatile-fact nhưng thiếu tool_result nguồn giảm về 0, grounding rate (số claim có citation / tổng claim volatile) tăng lên gần 100%.

3. Metric cần theo dõi: grounding rate (như trên), tỷ lệ trajectory bị Validator chặn và phải retry, độ trễ tăng thêm do bước validate, và số false-positive (Validator chặn nhầm câu thực sự không cần grounding)

4. Có thực sụ cần Agent không: Cho use case cụ thể này (lấy giá + nhân) - không cần Agent phức tạp, một pipeline cố định fetch_price -> multiply -> format là đủ và an toàn hơn nhiều so với LLM tự quyết có gọi tool hay không

5. Kiến trúc đơn giản hơn: nếu domain "volatile fact" có thể liệt kê hữu hạn trước (giá crypto, tỷ giá thời tiết) dùng workflow cố định (deterministic pipeline, không cần agent loop/planning tự do) sẽ loại bỏ hoàn toàn rủi ro hallucination -> chỉ nen dùng Agent thục sự khi domain/tool cần chọn động và không liệt kê trước được.